# LeetCode 2017 — Grid Game bằng Python

Link đề: https://leetcode.com/problems/grid-game/ · Độ khó: **Medium** · Chủ đề: Array, Matrix, Prefix Sum, Game Theory (minimax)

Skeleton LeetCode:

```python
class Solution:
    def gridGame(self, grid: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra rằng trên lưới `2 × n` chỉ đi **phải** hoặc **xuống**, mỗi đường đi được xác định bởi **đúng một cột rẽ xuống** `k`.
2. Phân tích trò chơi **minimax** hai bước: robot 1 chọn trước để **cực tiểu**, robot 2 chọn sau để **cực đại**.
3. Rút ra công thức: sau khi robot 1 rẽ ở cột `k`, robot 2 chỉ lấy được **max(phần còn lại hàng trên, phần còn lại hàng dưới)**.
4. Tính nhanh bằng **tổng tiền tố / hậu tố**, rồi tối ưu xuống `O(1)` bộ nhớ.
5. Thấy vì sao chiến lược "robot 1 tham lam lấy nhiều điểm nhất cho mình" là **sai**, và kiểm chứng bằng mô phỏng vét cạn.

## 2. Đề bài & ví dụ

Cho lưới `grid` kích thước `2 × n`, `grid[r][c]` là số điểm tại ô `(r, c)`. Hai robot cùng xuất phát ở `(0, 0)` và muốn tới `(1, n−1)`, mỗi bước chỉ đi **phải** `(r, c) → (r, c+1)` hoặc **xuống** `(r, c) → (r+1, c)`.

- Robot 1 đi trước, thu hết điểm trên đường đi và **đặt các ô đó về 0**.
- Sau đó robot 2 đi, thu điểm trên đường của nó (đường có thể giao với robot 1).

Robot 1 muốn **cực tiểu** điểm của robot 2; robot 2 muốn **cực đại** điểm của mình. Cả hai chơi tối ưu — trả về điểm của robot 2.

**Ví dụ 1:** `[[2,5,4],[1,5,1]]` → `4`.

**Ví dụ 2:** `[[3,3,1],[8,5,2]]` → `4`.

**Ví dụ 3:** `[[1,3,1,15],[1,3,3,1]]` → `7`.

**Ràng buộc:** `grid.length == 2`, `1 ≤ n ≤ 5·10⁴`, `1 ≤ grid[r][c] ≤ 10⁵`.

In [1]:
import random
import time
from itertools import accumulate
from typing import List

examples = [([[2, 5, 4], [1, 5, 1]], 4), ([[3, 3, 1], [8, 5, 2]], 4), ([[1, 3, 1, 15], [1, 3, 3, 1]], 7)]
for g, exp in examples:
    print(g[0]); print(g[1]); print("-> kỳ vọng", exp); print()

[2, 5, 4]
[1, 5, 1]
-> kỳ vọng 4

[3, 3, 1]
[8, 5, 2]
-> kỳ vọng 4

[1, 3, 1, 15]
[1, 3, 3, 1]
-> kỳ vọng 7



## 3. Kiến thức nền

### 3.1 Đường đi trên lưới 2 hàng = chọn cột rẽ

Vì chỉ đi phải/xuống và chỉ có 2 hàng, robot đi hàng trên từ cột `0` tới cột `k`, rẽ xuống tại cột `k`, rồi đi hàng dưới từ cột `k` tới `n−1`. Có đúng `n` đường đi (`k = 0..n−1`).

In [2]:
def path_cells(n, k):
    return [(0, c) for c in range(k + 1)] + [(1, c) for c in range(k, n)]

for k in range(3):
    print(f"k={k}:", path_cells(3, k))

k=0: [(0, 0), (1, 0), (1, 1), (1, 2)]
k=1: [(0, 0), (0, 1), (1, 1), (1, 2)]
k=2: [(0, 0), (0, 1), (0, 2), (1, 2)]


Một hàm vẽ nhỏ giúp **nhìn thấy** đường đi của hai robot (`1` = robot 1 đi qua, `2` = robot 2, `*` = cả hai). Ta sẽ dùng nó để xem lời giải tối ưu của ví dụ 1: robot 1 rẽ ở cột 1, robot 2 đi hết hàng trên.

In [3]:
def draw(grid, k1, k2):
    n = len(grid[0])
    p1, p2 = set(path_cells(n, k1)), set(path_cells(n, k2))
    for r in range(2):
        row = []
        for c in range(n):
            mark = "*" if (r, c) in p1 and (r, c) in p2 else "1" if (r, c) in p1 else "2" if (r, c) in p2 else "."
            row.append(f"{grid[r][c]:>3}[{mark}]")
        print(" ".join(row))

draw([[2, 5, 4], [1, 5, 1]], k1=1, k2=2)

  2[*]   5[*]   4[2]
  1[.]   5[1]   1[*]


### 3.2 Tổng tiền tố và hậu tố

`prefix[i] = a[0] + ... + a[i−1]` cho phép tính tổng mọi đoạn trong `O(1)`. "Phần hàng trên sau cột `k`" là tổng hậu tố `top[k+1..n−1]`; "phần hàng dưới trước cột `k`" là tổng tiền tố `bottom[0..k−1]`.

In [4]:
top = [1, 3, 1, 15]
P = [0] + list(accumulate(top))
print("prefix:", P, "| tổng top[2..3] =", P[4] - P[2], "| tổng top[0..1] =", P[2])

prefix: [0, 1, 4, 5, 20] | tổng top[2..3] = 16 | tổng top[0..1] = 4


### 3.3 Minimax: `min` của `max`

Trò chơi hai lượt: người thứ nhất chọn `k`, người thứ hai thấy `k` rồi chọn `j`. Giá trị trò chơi là `min_k max_j f(k, j)`. Nghĩa là: với mỗi `k`, giả sử đối thủ đáp trả tốt nhất (`max_j`), rồi chọn `k` cho kết quả tệ nhất của đối thủ là nhỏ nhất.

In [5]:
f = [[3, 7, 1], [4, 2, 5], [6, 6, 0]]       # f[k][j]
print("max theo j với mỗi k:", [max(row) for row in f], "-> min =", min(max(row) for row in f))

max theo j với mỗi k: [7, 5, 6] -> min = 5


### 3.4 Sao chép lưới để mô phỏng

Khi mô phỏng robot 1 "xóa" điểm, phải **sao chép** lưới (`[row[:] for row in grid]`) để không làm hỏng dữ liệu gốc cho các lần thử khác.

In [6]:
g = [[2, 5, 4], [1, 5, 1]]
g2 = [row[:] for row in g]
for r, c in path_cells(3, 1):
    g2[r][c] = 0
print("gốc:", g, "| sau robot 1 (k=1):", g2)

gốc: [[2, 5, 4], [1, 5, 1]] | sau robot 1 (k=1): [[0, 0, 4], [1, 0, 0]]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 3: `top = [1,3,1,15]`, `bottom = [1,3,3,1]`. Thử từng cột rẽ `k` của robot 1:

| k | robot 1 xóa | còn lại hàng trên (sau k) | còn lại hàng dưới (trước k) | robot 2 lấy được |
|---|---|---|---|---|
| 0 | top[0], bottom[0..3] | 3+1+15 = 19 | 0 | 19 |
| 1 | top[0..1], bottom[1..3] | 1+15 = 16 | 1 | 16 |
| 2 | top[0..2], bottom[2..3] | 15 | 1+3 = 4 | 15 |
| 3 | top[0..3], bottom[3] | 0 | 1+3+3 = 7 | **7** |

Vì sao robot 2 chỉ lấy được **một** trong hai phần? Phần còn lại ở hàng trên nằm **bên phải** cột `k`, phần còn lại ở hàng dưới nằm **bên trái** cột `k`. Robot 2 muốn lấy hàng dưới bên trái thì phải xuống sớm (trước `k`), khi đó không quay lại hàng trên được nữa. Vậy nó chọn phần lớn hơn.

Robot 1 chọn `k = 3` ⇒ đáp án `7`.

In [7]:
top, bot = [1, 3, 1, 15], [1, 3, 3, 1]
n = len(top)
for k in range(n):
    up, down = sum(top[k + 1:]), sum(bot[:k])
    print(f"k={k}: còn trên={up:2d}, còn dưới={down:2d} -> robot 2 lấy {max(up, down)}")

k=0: còn trên=19, còn dưới= 0 -> robot 2 lấy 19
k=1: còn trên=16, còn dưới= 1 -> robot 2 lấy 16
k=2: còn trên=15, còn dưới= 4 -> robot 2 lấy 15
k=3: còn trên= 0, còn dưới= 7 -> robot 2 lấy 7


## 5. Cách 1 — Brute Force / baseline: mô phỏng đầy đủ hai robot

**Ý tưởng:** với mỗi `k` của robot 1: sao chép lưới, xóa các ô trên đường; với mỗi `j` của robot 2: cộng điểm trên đường. Lấy `min_k max_j`.

In [8]:
def grid_game_brute(grid):
    n = len(grid[0])
    best = float("inf")
    for k in range(n):
        g = [row[:] for row in grid]
        for r, c in path_cells(n, k):
            g[r][c] = 0
        robot2 = max(sum(g[r][c] for r, c in path_cells(n, j)) for j in range(n))
        best = min(best, robot2)
    return best

for g, exp in examples:
    got = grid_game_brute(g); print(got, "kỳ vọng", exp); assert got == exp

4 kỳ vọng 4
4 kỳ vọng 4
7 kỳ vọng 7


**Độ phức tạp:** `n` lựa chọn `k` × `n` lựa chọn `j` × `O(n)` để cộng một đường ⇒ `O(n³)`. Với `n = 5·10⁴` là `10¹⁴` — không thể. Space `O(n)`.

**Nút thắt:** (1) với `k` cố định, ta không cần thử mọi `j` — kết quả luôn là `max(phần trên, phần dưới)`; (2) các tổng đoạn có thể tính trong `O(1)` bằng tiền tố.

In [9]:
rng = random.Random(0)
for n in (20, 40, 80):
    g = [[rng.randint(1, 9) for _ in range(n)] for _ in range(2)]
    t = time.perf_counter(); grid_game_brute(g)
    print(f"n={n}: {time.perf_counter() - t:.3f}s (gấp đôi n -> ~x8)")

n=20: 0.001s (gấp đôi n -> ~x8)
n=40: 0.004s (gấp đôi n -> ~x8)
n=80: 0.028s (gấp đôi n -> ~x8)


## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề (đáp trả tốt nhất của robot 2):** sau khi robot 1 rẽ tại `k`, các ô còn điểm là `top[k+1..n−1]` và `bottom[0..k−1]`. Robot 2 rẽ tại `j`:
- **nếu** `j ≤ k` **thì** nó lấy được `bottom[j..k−1]` (hàng trên từ `0..j` đã bị xóa, hàng dưới từ `k` trở đi đã bị xóa) — tối đa khi `j = 0`: cả `bottom[0..k−1]`;
- **nếu** `j ≥ k` **thì** nó lấy được `top[k+1..j]` — tối đa khi `j = n−1`: cả `top[k+1..n−1]`.

Vậy `robot2(k) = max(top_suffix(k+1), bottom_prefix(k))`.

**Công thức:**
$$\text{đáp án} = \min_{0 \le k < n} \max\Big(\sum_{c > k} top[c],\ \sum_{c < k} bottom[c]\Big).$$

**Invariant khi quét `k` tăng dần:** `up = Σ_{c>k} top[c]` giảm dần (trừ `top[k]` mỗi bước), `down = Σ_{c<k} bottom[c]` tăng dần (cộng `bottom[k−1]`). Không cần mảng.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Dùng bổ đề nhưng tính tổng bằng `sum` (`O(n²)`)

In [10]:
def grid_game_quadratic(grid):
    top, bot = grid
    n = len(top)
    return min(max(sum(top[k + 1:]), sum(bot[:k])) for k in range(n))

for g, exp in examples:
    assert grid_game_quadratic(g) == exp
print([grid_game_quadratic(g) for g, _ in examples])

[4, 4, 7]


**Độ phức tạp Cách 2:** `O(n²)` — với `n = 5·10⁴` khoảng `2,5·10⁹` phép cộng, vẫn quá chậm cho Python.

### 7.2 Cách 3 — Mảng tổng tiền tố (`O(n)` thời gian, `O(n)` bộ nhớ)

**Giải thích từng đoạn:**
1. `PT[i] = top[0] + ... + top[i−1]`, `PB` tương tự cho hàng dưới.
2. `up(k) = PT[n] − PT[k+1]`, `down(k) = PB[k]`.
3. Đáp án `min_k max(up(k), down(k))`.

In [11]:
def grid_game_prefix(grid):
    top, bot = grid
    n = len(top)
    PT = [0] + list(accumulate(top))
    PB = [0] + list(accumulate(bot))
    return min(max(PT[n] - PT[k + 1], PB[k]) for k in range(n))

for g, exp in examples:
    assert grid_game_prefix(g) == exp
print([grid_game_prefix(g) for g, _ in examples])

[4, 4, 7]


In [12]:
class SolutionPrefix:
    def gridGame(self, grid: List[List[int]]) -> int:
        top, bot = grid
        n = len(top)
        PT = [0] + list(accumulate(top))
        PB = [0] + list(accumulate(bot))
        return min(max(PT[n] - PT[k + 1], PB[k]) for k in range(n))

print([SolutionPrefix().gridGame(g) for g, _ in examples])

[4, 4, 7]


**Độ phức tạp Cách 3:** Time `O(n)`, Space `O(n)`.

### 7.3 Cách 4 — Hai biến chạy (`O(1)` bộ nhớ)

**Giải thích từng đoạn:**
1. `up = sum(top)`, `down = 0`.
2. Với `k = 0..n−1`: `up −= top[k]` (giờ `up` = tổng hàng trên **sau** `k`); cập nhật đáp án `min(ans, max(up, down))`; rồi `down += bot[k]` (chuẩn bị cho `k+1`).

In [13]:
def grid_game_o1(grid):
    up, down = sum(grid[0]), 0
    ans = float("inf")
    for k in range(len(grid[0])):
        up -= grid[0][k]
        ans = min(ans, max(up, down))
        down += grid[1][k]
    return ans

for g, exp in examples:
    assert grid_game_o1(g) == exp
print([grid_game_o1(g) for g, _ in examples])

[4, 4, 7]


**Độ phức tạp Cách 4:** Time `O(n)`, Space `O(1)`.

**Nhận xét thêm:** `up(k)` giảm và `down(k)` tăng theo `k`, nên `max(up, down)` là hàm "chữ V" (giảm rồi tăng) ⇒ có thể **tìm nhị phân** điểm cắt `up ≈ down`. Không cần thiết vì đã `O(n)`, nhưng là một góc nhìn hay về hình dạng hàm mục tiêu.

### 7.4 Cách 5 — Tìm nhị phân trên hàm "chữ V"

Vì mọi `grid[r][c] ≥ 1`: `up(k)` **giảm nghiêm ngặt**, `down(k)` **tăng nghiêm ngặt** theo `k`. Gọi `k*` là chỉ số nhỏ nhất có `down(k*) ≥ up(k*)`. Với `k < k*`, `max = up(k)` (đang giảm); với `k ≥ k*`, `max = down(k)` (đang tăng). Vậy cực tiểu nằm ở `k*` hoặc `k* − 1`. Tìm `k*` bằng tìm nhị phân trên mảng tiền tố.

Cách này không nhanh hơn về bậc (vẫn cần `O(n)` để dựng tiền tố), nhưng nếu tiền tố đã có sẵn và có **nhiều truy vấn** thì mỗi truy vấn chỉ `O(log n)`.

In [14]:
def grid_game_bsearch(grid):
    top, bot = grid; n = len(top)
    PT = [0] + list(accumulate(top)); PB = [0] + list(accumulate(bot))
    up = lambda k: PT[n] - PT[k + 1]
    down = lambda k: PB[k]
    lo, hi = 0, n - 1                       # tìm k nhỏ nhất có down(k) >= up(k)
    while lo < hi:
        mid = (lo + hi) // 2
        if down(mid) >= up(mid):
            hi = mid
        else:
            lo = mid + 1
    cands = [lo] + ([lo - 1] if lo > 0 else [])
    return min(max(up(k), down(k)) for k in cands)

for g, exp in examples:
    assert grid_game_bsearch(g) == exp
print([grid_game_bsearch(g) for g, _ in examples])

[4, 4, 7]


In [15]:
# Kiểm chứng hình dạng "chữ V" của f(k) = max(up(k), down(k)) trên dữ liệu ngẫu nhiên
rng = random.Random(77)
for _ in range(300):
    n = rng.randint(1, 30); g = [[rng.randint(1, 50) for _ in range(n)] for _ in range(2)]
    f = [max(sum(g[0][k + 1:]), sum(g[1][:k])) for k in range(n)]
    m = f.index(min(f))
    assert all(f[i] >= f[i + 1] for i in range(m))          # không tăng trước điểm cực tiểu
    assert all(f[i] <= f[i + 1] for i in range(m, n - 1))   # không giảm sau điểm cực tiểu
    assert grid_game_bsearch(g) == min(f)
print("f(k) giảm rồi tăng; tìm nhị phân khớp trên 300 lưới ngẫu nhiên")

f(k) giảm rồi tăng; tìm nhị phân khớp trên 300 lưới ngẫu nhiên


**Độ phức tạp Cách 5:** dựng tiền tố `O(n)`, tìm nhị phân `O(log n)`. Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 4 trên ví dụ 2 `[[3,3,1],[8,5,2]]`. Ban đầu `up = 7`, `down = 0`.

| k | up sau khi trừ top[k] | down | max(up, down) | ans | down sau khi cộng bot[k] |
|---|---|---|---|---|---|
| 0 | 7 − 3 = 4 | 0 | 4 | 4 | 8 |
| 1 | 4 − 3 = 1 | 8 | 8 | 4 | 13 |
| 2 | 1 − 1 = 0 | 13 | 13 | 4 | 15 |

Đáp án `4` (robot 1 rẽ ngay ở cột 0).

In [16]:
def trace_o1(grid):
    up, down = sum(grid[0]), 0
    ans = float("inf")
    print(f"ban đầu: up={up}, down={down}")
    for k in range(len(grid[0])):
        up -= grid[0][k]
        cur = max(up, down)
        ans = min(ans, cur)
        print(f"k={k}: up={up:3d}, down={down:3d}, robot 2 lấy max={cur:3d}, ans={ans}")
        down += grid[1][k]
    return ans

print("Đáp án:", trace_o1([[3, 3, 1], [8, 5, 2]]))
print()
print("Đáp án:", trace_o1([[1, 3, 1, 15], [1, 3, 3, 1]]))

ban đầu: up=7, down=0
k=0: up=  4, down=  0, robot 2 lấy max=  4, ans=4
k=1: up=  1, down=  8, robot 2 lấy max=  8, ans=4
k=2: up=  0, down= 13, robot 2 lấy max= 13, ans=4
Đáp án: 4

ban đầu: up=20, down=0
k=0: up= 19, down=  0, robot 2 lấy max= 19, ans=19
k=1: up= 16, down=  1, robot 2 lấy max= 16, ans=16
k=2: up= 15, down=  4, robot 2 lấy max= 15, ans=15
k=3: up=  0, down=  7, robot 2 lấy max=  7, ans=7
Đáp án: 7


## 9. Code nộp LeetCode

In [17]:
class Solution:
    def gridGame(self, grid: List[List[int]]) -> int:
        top, bot = grid
        up, down = sum(top), 0           # up = hàng trên sau cột k, down = hàng dưới trước cột k
        ans = float("inf")
        for k in range(len(top)):        # robot 1 rẽ xuống tại cột k
            up -= top[k]
            ans = min(ans, max(up, down))   # robot 2 lấy phần lớn hơn
            down += bot[k]
        return ans

for g, exp in examples:
    assert Solution().gridGame(g) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (đường đi).** Mỗi đường đi hợp lệ trên lưới `2 × n` tương ứng duy nhất một cột rẽ `k ∈ [0, n−1]` — vì phải xuống hàng dưới đúng một lần.

**Bổ đề 2 (đáp trả của robot 2).** Cố định `k`. Sau khi robot 1 đi, ô `(0, c)` còn điểm ⇔ `c > k`; ô `(1, c)` còn điểm ⇔ `c < k`. Robot 2 với cột rẽ `j` thu được `Σ_{k<c≤j} top[c] + Σ_{j≤c<k} bottom[c]`. Nếu `j ≤ k` thì tổng đầu rỗng; nếu `j ≥ k` thì tổng sau rỗng — **không thể** có cả hai tổng khác rỗng. Vì mọi điểm đều dương, tối ưu là `j = 0` (lấy hết `bottom[0..k−1]`) hoặc `j = n−1` (lấy hết `top[k+1..n−1]`). Vậy `robot2(k) = max(up(k), down(k))`.

**Kết luận.** Robot 1 biết robot 2 sẽ đáp trả tối ưu, nên chọn `k` cực tiểu `robot2(k)`. Cách 4 duyệt mọi `k` và duy trì đúng `up(k)`, `down(k)` (bất biến ở mục 6) ⇒ trả về đúng `min_k max(up, down)`.

Kiểm chứng thực nghiệm **Bổ đề 2**: với mọi `k`, đáp trả tốt nhất của robot 2 luôn đạt được tại `j = 0` hoặc `j = n − 1`, và bằng `max(up(k), down(k))`.

In [18]:
rng = random.Random(10)
for _ in range(300):
    n = rng.randint(1, 10); grid = [[rng.randint(1, 20) for _ in range(n)] for _ in range(2)]
    for k in range(n):
        g = [row[:] for row in grid]
        for r, c in path_cells(n, k): g[r][c] = 0
        vals = [sum(g[r][c] for r, c in path_cells(n, j)) for j in range(n)]
        assert max(vals) == max(vals[0], vals[-1]) == max(sum(grid[0][k + 1:]), sum(grid[1][:k]))
print("Bổ đề 2 đúng trên 300 lưới ngẫu nhiên (mọi k)")

Bổ đề 2 đúng trên 300 lưới ngẫu nhiên (mọi k)


## 11. Phân tích Time / Space

Gọi `n` = số cột.

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng | `O(n³)` | `O(n)` |
| 2. Bổ đề + `sum` | `O(n²)` | `O(n)` (do slice) |
| 3. Tiền tố | `O(n)` | `O(n)` |
| 4. Hai biến | `O(n)` | `O(1)` |
| 5. Tìm nhị phân | `O(n)` + `O(log n)` | `O(n)` |

Đáp án có thể tới `5·10⁴ · 10⁵ = 5·10⁹` > `2³¹` — Python không lo, Java/C++ cần `long`.

In [19]:
rng = random.Random(2017)
big = [[rng.randint(1, 10**5) for _ in range(5 * 10**4)] for _ in range(2)]
worst = [[10**5] * (5 * 10**4), [10**5] * (5 * 10**4)]
for name, fn in [("prefix", grid_game_prefix), ("o1", grid_game_o1), ("Solution", lambda g: Solution().gridGame(g))]:
    t = time.perf_counter(); r1 = fn(big); r2 = fn(worst)
    print(f"{name:8s}: ngẫu nhiên={r1}, toàn 10^5={r2} ({time.perf_counter() - t:.3f}s)")
print("2^31 - 1 =", 2**31 - 1)

prefix  : ngẫu nhiên=1250207775, toàn 10^5=2500000000 (0.016s)
o1      : ngẫu nhiên=1250207775, toàn 10^5=2500000000 (0.011s)
Solution: ngẫu nhiên=1250207775, toàn 10^5=2500000000 (0.009s)
2^31 - 1 = 2147483647


## 12. Bộ test

Ví dụ đề; biên `n = 1` (robot 2 luôn được 0), `n = 2`; mảng tăng/giảm; đối chiếu mọi cách với mô phỏng vét cạn trên lưới ngẫu nhiên (seed cố định).

In [20]:
fns = {
    "brute": grid_game_brute, "quadratic": grid_game_quadratic, "prefix": grid_game_prefix, "o1": grid_game_o1,
    "bsearch": grid_game_bsearch,
    "Solution": lambda g: Solution().gridGame(g), "SolutionPrefix": lambda g: SolutionPrefix().gridGame(g),
}
edge = [([[5], [7]], 0), ([[1, 2], [3, 4]], 2), ([[10, 1], [1, 10]], 1), ([[1, 1, 1, 1], [1, 1, 1, 1]], 2)]
for g, exp in examples + edge:
    for name, fn in fns.items():
        assert fn(g) == exp, (name, g, fn(g))
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [21]:
rng = random.Random(42)
for _ in range(500):
    n = rng.randint(1, 12)
    hi = rng.choice([3, 10, 100000])
    g = [[rng.randint(1, hi) for _ in range(n)] for _ in range(2)]
    want = grid_game_brute(g)
    for name, fn in fns.items():
        assert fn(g) == want, (name, g)
print("500 test ngẫu nhiên: mọi cách khớp mô phỏng vét cạn")

500 test ngẫu nhiên: mọi cách khớp mô phỏng vét cạn


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Robot 1 tham lam: chọn đường có **nhiều điểm nhất cho mình**

Lấy nhiều điểm nhất không đồng nghĩa để lại ít nhất cho robot 2 (vì robot 2 chỉ lấy **một** trong hai phần còn lại). Tìm phản ví dụ bằng máy:

In [22]:
def wrong_greedy_self(grid):
    top, bot = grid; n = len(top)
    k = max(range(n), key=lambda k: sum(top[:k + 1]) + sum(bot[k:]))   # SAI: tối đa điểm robot 1
    return max(sum(top[k + 1:]), sum(bot[:k]))

rng = random.Random(1)
for _ in range(100000):
    n = rng.randint(2, 4)
    g = [[rng.randint(1, 9) for _ in range(n)] for _ in range(2)]
    if wrong_greedy_self(g) != Solution().gridGame(g):
        break
print("phản ví dụ:", g, "| sai:", wrong_greedy_self(g), "| đúng:", Solution().gridGame(g))

phản ví dụ: [[7, 1, 8], [5, 4, 2]] | sai: 9 | đúng: 8


In [23]:
print("sửa (minimax):", grid_game_o1(g))

sửa (minimax): 8


### Lỗi 2 — Cộng cả hai phần còn lại thay vì lấy `max`

Robot 2 **không thể** lấy cả phần trên (bên phải `k`) và phần dưới (bên trái `k`).

In [24]:
def wrong_sum_both(grid):
    top, bot = grid; up, down, ans = sum(top), 0, float("inf")
    for k in range(len(top)):
        up -= top[k]
        ans = min(ans, up + down)                 # SAI: cộng cả hai
        down += bot[k]
    return ans

g = [[2, 5, 4], [1, 5, 1]]
print("sai:", wrong_sum_both(g), "| đúng:", Solution().gridGame(g))

sai: 5 | đúng: 4


In [25]:
print("sửa (max):", grid_game_o1(g))

sửa (max): 4


### Lỗi 3 — Lệch chỉ số: tính phần hàng trên **bao gồm** cột `k`

Cột `k` ở hàng trên đã bị robot 1 đi qua (robot 1 rẽ **tại** `k`). Nếu cập nhật `up` **sau** khi tính `max`, ta tính dư `top[k]`.

In [26]:
def wrong_order(grid):
    top, bot = grid; up, down, ans = sum(top), 0, float("inf")
    for k in range(len(top)):
        ans = min(ans, max(up, down))             # SAI: up còn chứa top[k]
        up -= top[k]
        down += bot[k]
    return ans

g = [[2, 5, 4], [1, 5, 1]]
print("sai:", wrong_order(g), "| đúng:", Solution().gridGame(g))

sai: 6 | đúng: 4


In [27]:
print("sửa (trừ top[k] trước):", grid_game_o1(g))

sửa (trừ top[k] trước): 4


### Lỗi 4 — Khởi tạo `ans = 0` cho bài toán tìm `min`

`min(0, ...)` luôn ra `0`. Phải khởi tạo `ans = float('inf')` (hoặc giá trị của `k = 0`).

In [28]:
def wrong_init(grid):
    top, bot = grid; up, down, ans = sum(top), 0, 0     # SAI: ans = 0
    for k in range(len(top)):
        up -= top[k]; ans = min(ans, max(up, down)); down += bot[k]
    return ans

print("sai:", [wrong_init(g) for g, _ in examples], "| đúng:", [e for _, e in examples])

sai: [0, 0, 0] | đúng: [4, 4, 7]


In [29]:
print("sửa:", [grid_game_o1(g) for g, _ in examples])

sửa: [4, 4, 7]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Lưới 2 hàng, đi phải/xuống** ⇒ đường đi = một cột rẽ ⇒ thường quy về **tiền tố/hậu tố**.
- **Hai người chơi, lần lượt, đối nghịch** ⇒ minimax: cố định nước của người đi trước, tính **đáp trả tốt nhất** của người sau (thường có công thức đóng), rồi tối ưu người đi trước.
- "Một tăng, một giảm" ⇒ `max(up, down)` có dạng chữ V ⇒ `min` của nó nằm gần điểm cắt.
- Đừng nhầm "tối thiểu điểm đối thủ" với "tối đa điểm của mình".
- Bài có `n` đến `5·10⁴` và chỉ 2 hàng ⇒ chắc chắn `O(n)`.

## 15. Biến thể của bài

1. **Robot 1 muốn tối đa (điểm mình − điểm robot 2)** (bài tập 5) — vẫn duyệt `k`, công thức khác.
2. **Lưới `m × n` tổng quát**: robot 2 có nhiều lựa chọn hơn; cần DP lưới cho robot 2 với mỗi đường của robot 1 — bài khó hơn nhiều.
3. **LeetCode 741 — Cherry Pickup (Hard):** hai đường đi **đồng thời** cùng cực đại tổng (không đối nghịch) — DP 3 chiều.
4. **LeetCode 64 — Minimum Path Sum:** một robot, lưới tổng quát — DP cơ bản.
5. **Hai robot, robot 2 muốn tối thiểu, robot 1 muốn tối đa điểm robot 2** — đổi `min`/`max` (ví dụ dưới).

In [30]:
def friendly_game(grid):
    """Robot 1 muốn robot 2 được NHIỀU nhất (hợp tác): max_k max(up, down)."""
    up, down, ans = sum(grid[0]), 0, 0
    for k in range(len(grid[0])):
        up -= grid[0][k]; ans = max(ans, max(up, down)); down += grid[1][k]
    return ans

def friendly_brute(grid):
    n = len(grid[0]); best = 0
    for k in range(n):
        g = [row[:] for row in grid]
        for r, c in path_cells(n, k): g[r][c] = 0
        best = max(best, max(sum(g[r][c] for r, c in path_cells(n, j)) for j in range(n)))
    return best

rng = random.Random(9)
for _ in range(200):
    n = rng.randint(1, 8); g = [[rng.randint(1, 9) for _ in range(n)] for _ in range(2)]
    assert friendly_game(g) == friendly_brute(g)
print("Biến thể 'hợp tác' (tự dựng) khớp mô phỏng; ví dụ 3 ->", friendly_game(examples[2][0]))

Biến thể 'hợp tác' (tự dựng) khớp mô phỏng; ví dụ 3 -> 19


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | thử mọi `k`, mọi `j` | `O(n³)` | `O(n)` | kiểm tra |
| 2. Bổ đề + `sum` | `max(top sau k, bottom trước k)` | `O(n²)` | `O(n)` | hiểu công thức |
| 3. Tiền tố | mảng `PT`, `PB` | `O(n)` | `O(n)` | rõ ràng |
| 4. Hai biến | `up` giảm, `down` tăng | `O(n)` | `O(1)` | **nộp bài** |
| 5. Tìm nhị phân | điểm cắt của hàm chữ V | `O(n)` dựng + `O(log n)` | `O(n)` | nhiều truy vấn trên cùng tiền tố |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `gridGame([[5],[7]])` và `gridGame([[1,1],[1,1]])`.
2. **Tính tay:** `gridGame([[1,2,3,4],[4,3,2,1]])` — lập bảng `k, up, down, max`.
3. **Điền TODO:** `best_turn(grid)` trả về **cột rẽ** `k` tối ưu của robot 1 (nếu nhiều, lấy `k` nhỏ nhất).
4. **Tự viết lại:** viết lại Cách 4 không nhìn code.
5. **Mini challenge:** `score_diff(grid)` — robot 1 muốn **cực đại** `(điểm robot 1) − (điểm robot 2)`, robot 2 vẫn cực đại điểm của mình. Kiểm bằng mô phỏng.

In [31]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def best_turn(grid):
    # TODO
    pass

In [32]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_grid_game(grid):
    # TODO: up = sum(top), down = 0, duyệt k
    pass

In [33]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def score_diff(grid):
    # TODO: điểm robot 1 với cột k = PT[k+1] + (PB[n] - PB[k])
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) `n = 1`: robot 1 đi hết lưới. (2) `[[1,1],[1,1]]`: `k = 0` ⇒ `max(1, 0)`; `k = 1` ⇒ `max(0, 1)`. (3) Cả hai đều 1.

**Bài 2.** (1) `up` sau từng bước: `9, 7, 4, 0`. (2) `down` trước từng bước: `0, 4, 7, 9`. (3) `max`: `9, 7, 7, 9` ⇒ min `7`.

**Bài 3.** (1) Như Cách 4 nhưng lưu `k` khi `max(up, down) < ans`. (2) Dùng `<` (không phải `≤`) để giữ `k` nhỏ nhất. (3) Kiểm: `max(up(k), down(k)) == gridGame(grid)`.

**Bài 4.** (1) Trừ `top[k]` trước. (2) Lấy `max`, cập nhật `min`. (3) Cộng `bot[k]` sau.

**Bài 5.** (1) Robot 2 vẫn đáp trả `max(up, down)`. (2) Điểm robot 1 với `k`: tổng `top[0..k] + bot[k..n−1]`. (3) Lấy `max_k` của hiệu.

## 19. Đáp án tham khảo

In [34]:
# Bài 1, 2
assert Solution().gridGame([[5], [7]]) == 0 and Solution().gridGame([[1, 1], [1, 1]]) == 1
assert Solution().gridGame([[1, 2, 3, 4], [4, 3, 2, 1]]) == 7
print("Bài 1: 0 và 1 | Bài 2: 7")
trace_o1([[1, 2, 3, 4], [4, 3, 2, 1]])

Bài 1: 0 và 1 | Bài 2: 7
ban đầu: up=10, down=0
k=0: up=  9, down=  0, robot 2 lấy max=  9, ans=9
k=1: up=  7, down=  4, robot 2 lấy max=  7, ans=7
k=2: up=  4, down=  7, robot 2 lấy max=  7, ans=7
k=3: up=  0, down=  9, robot 2 lấy max=  9, ans=7


7

In [35]:
# Bài 3, 4
def best_turn_ans(grid):
    up, down, ans, best_k = sum(grid[0]), 0, float("inf"), -1
    for k in range(len(grid[0])):
        up -= grid[0][k]
        if max(up, down) < ans:
            ans, best_k = max(up, down), k
        down += grid[1][k]
    return best_k

my_grid_game_ans = grid_game_o1
rng = random.Random(3)
for _ in range(300):
    n = rng.randint(1, 10); g = [[rng.randint(1, 9) for _ in range(n)] for _ in range(2)]
    k = best_turn_ans(g)
    assert max(sum(g[0][k + 1:]), sum(g[1][:k])) == Solution().gridGame(g) == my_grid_game_ans(g)
    assert all(max(sum(g[0][j + 1:]), sum(g[1][:j])) > Solution().gridGame(g) for j in range(k))
print("Bài 3, 4: OK; ví dụ 3 -> k =", best_turn_ans(examples[2][0]))

Bài 3, 4: OK; ví dụ 3 -> k = 3


In [36]:
# Bài 5
def score_diff_ans(grid):
    top, bot = grid; n = len(top)
    PT = [0] + list(accumulate(top)); PB = [0] + list(accumulate(bot))
    best = float("-inf")
    for k in range(n):
        r1 = PT[k + 1] + PB[n] - PB[k]
        r2 = max(PT[n] - PT[k + 1], PB[k])
        best = max(best, r1 - r2)
    return best

def score_diff_brute(grid):
    n = len(grid[0]); best = float("-inf")
    for k in range(n):
        r1 = sum(grid[r][c] for r, c in path_cells(n, k))
        g = [row[:] for row in grid]
        for r, c in path_cells(n, k): g[r][c] = 0
        r2 = max(sum(g[r][c] for r, c in path_cells(n, j)) for j in range(n))
        best = max(best, r1 - r2)
    return best

rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 9); g = [[rng.randint(1, 9) for _ in range(n)] for _ in range(2)]
    assert score_diff_ans(g) == score_diff_brute(g)
print("Bài 5: OK; ví dụ 1 ->", score_diff_ans(examples[0][0]))

Bài 5: OK; ví dụ 1 -> 9


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "lưới 2 hàng + minimax":**

```python
up, down, ans = sum(top), 0, float("inf")
for k in range(n):           # người đi trước rẽ tại k
    up -= top[k]             # phần trên còn lại (sau k)
    ans = min(ans, max(up, down))   # người sau lấy phần lớn hơn
    down += bot[k]           # phần dưới còn lại (trước k+1)
return ans
```

In [37]:
def template(top, bot):
    up, down, ans = sum(top), 0, float("inf")
    for k in range(len(top)):
        up -= top[k]; ans = min(ans, max(up, down)); down += bot[k]
    return ans
print([template(*g) for g, _ in examples])

[4, 4, 7]


**Checklist:**

- [ ] Robot 2 lấy **max** của hai phần (không cộng).
- [ ] Phần trên còn lại là **sau** cột `k` (trừ `top[k]` trước khi tính).
- [ ] Phần dưới còn lại là **trước** cột `k` (cộng `bot[k]` sau khi tính).
- [ ] `ans` khởi tạo `inf` cho bài `min`.
- [ ] Không tham lam tối đa điểm robot 1.
- [ ] Đã thử `n = 1`, `n = 2`, ví dụ đề.